# pycograd — batching with `vmap`

This is the **batching** companion to `pycograd_demo.ipynb`. Same models, same
pipeline style — but every forward is written for a **single example**, and
[`vmap`](https://jax.readthedocs.io/en/latest/_autosummary/jax.vmap.html) vectorizes
it over the batch in one pass (no Python loop, no hand-broadcasting):

```python
from pycograd import vmap

forward = $ |> ...              # written for ONE example
batched = vmap(forward)        # maps over axis 0; weights stay shared
objective = lambda: batched(X) |> loss($, target)
```

`vmap` is one level of pycograd's trace-level interpreter stack, so it **composes**
with `grad`: `weights.grad(objective)` differentiates straight through the batched
pass. Weights are captured ambiently (not batched), exactly like passing
`in_axes=None` for them. The headline win is the two sequence models at the end,
where the explicit `for`-loop over examples collapses into a single `vmap`.

**Requires** the `pipescript` extra: `pip install pycograd[notebook]`.

In [1]:
%load_ext pycograd

## The primitive: `grad`

Underneath the DSL, `grad` / `value_and_grad` differentiate an ordinary numpy
function — the array argument is lifted onto the tape for you.

In [2]:
import numpy as np
from pycograd import grad

def f(x):
    return np.sum(np.sin(x * x))

x = np.array([0.5, 1.0, 1.5])
(gx,) = grad(f)(x)
print("autodiff :", np.round(gx, 4))
print("analytic :", np.round(2 * x * np.cos(x * x), 4))

autodiff : [ 0.9689  1.0806 -1.8845]
analytic : [ 0.9689  1.0806 -1.8845]


## The primitive: `vmap`

`vmap(g)` turns a function written for a **single** input into one that runs over a
whole batch — vectorized (one pass), not looped. Here `g` maps a length-3 vector to a
scalar; `vmap(g)` maps a `(B, 3)` batch to a `(B,)` vector of scalars, matching a
Python loop exactly but without one. Use `in_axes` to mark which arguments are
batched (an int axis) versus shared across the batch (`None`).

In [3]:
from pycograd import vmap

def g(v):                      # one length-3 vector -> one scalar
    return np.sum(np.sin(v * v))

batch = np.array([[0.5, 1.0, 1.5],
                  [0.2, 0.4, 0.6],
                  [1.0, 1.0, 1.0]])

vectorized = vmap(g)(batch)                      # one batched pass
looped     = np.array([g(v) for v in batch])     # the Python-loop oracle
print("vmap   :", np.round(vectorized, 4))
print("loop   :", np.round(looped, 4))

# vmap composes with grad: per-sample gradients, stacked over the batch.
per_sample = vmap(grad(g))(batch)
print("per-sample grads:\n", np.round(per_sample, 4))

vmap   : [1.8669 0.5516 2.5244]
loop   : [1.8669 0.5516 2.5244]
per-sample grads:
 [[[ 0.9689  1.0806 -1.8845]
  [ 0.3997  0.7898  1.1231]
  [ 1.0806  1.0806  1.0806]]]


## Setup: data and a loss

There is no op library to import — pycograd differentiates raw numpy. Activations are
just numpy used as pipe stages: `relu` is `np.maximum(0.0, $)`, and a linear layer is
`$ @ w + b`.

The one thing worth naming is the **loss**. `softmax_ce` takes a `(batch, classes)`
block of logits and the matching one-hot targets, and returns the mean cross-entropy.

In [4]:
def softmax_ce(logits, onehot):
    z = logits - np.max(logits, axis=1, keepdims=True)          # stabilize
    logp = z - np.log(np.sum(np.exp(z), axis=1, keepdims=True)) # log-softmax
    return -np.mean(np.sum(onehot * logp, axis=1))

rng = np.random.default_rng(0)
m = 40
centers = np.array([[2.0, 2.0], [-2.0, 2.0], [0.0, -2.5]])
X = np.vstack([rng.normal(c, 0.5, (m, 2)) for c in centers])
labels = np.repeat(np.arange(3), m)
Y = np.eye(3)[labels]   # one-hot, 3 classes

## Example 1 — a linear softmax classifier

The per-example forward is `$ @ w + b`: it maps one 2-D point to 3 logits. `vmap`
lifts it to the whole `(N, 2)` batch, producing `(N, 3)` logits — the weights `w`, `b`
are captured ambiently, so they are shared across the batch (not mapped).

In [5]:
with params{
    w = np.zeros((2, 3))
    b = np.zeros(3)
} as weights:
    forward = $ |> $ @ w + b           # one (2,) point -> (3,) logits
    batched = vmap(forward)            # (N, 2) -> (N, 3)
    objective = lambda: batched(X) |> softmax_ce($, Y)
    first = last = None
    for _ in range(200):
        value, grads = weights.grad(objective)
        first = float(value) if first is None else first
        last = float(value)
        weights.step(grads, 0.5)
    logits = batched(X)

print("loss %.4f -> %.4f" % (first, last))
print("train accuracy:", np.mean(np.argmax(logits, axis=1) == labels))

loss 1.0986 -> 0.0040
train accuracy: 1.0


## Example 2 — a 2-layer MLP, as one pipeline

Same idea: the pipeline `$ @ w1 + b1 |> relu |> $ @ w2 + b2` describes **one**
example's path through the net; `vmap` runs it over the batch. `relu` is inlined as
the pipe stage `np.maximum(0.0, $)`.

In [6]:
with params{
    w1 = 0.3 * rng.standard_normal((2, 16))
    b1 = np.zeros(16)
    w2 = 0.3 * rng.standard_normal((16, 3))
    b2 = np.zeros(3)
} as weights:
    forward = $ |> $ @ w1 + b1 |> np.maximum(0.0, $) |> $ @ w2 + b2
    batched = vmap(forward)
    objective = lambda: batched(X) |> softmax_ce($, Y)
    first = last = None
    for _ in range(300):
        value, grads = weights.grad(objective)
        first = float(value) if first is None else first
        last = float(value)
        weights.step(grads, 0.5)
    logits = batched(X)

print("loss %.4f -> %.4f" % (first, last))
print("train accuracy:", np.mean(np.argmax(logits, axis=1) == labels))

loss 1.5142 -> 0.0007
train accuracy: 1.0


## Example 3 — a frozen parameter

`frozen[...]` holds a weight fixed: its gradient comes back `None` and `weights.step`
skips it. Here the first-layer bias never moves while everything else trains — and the
forward is still written per-example and `vmap`'d over the batch.

In [7]:
with params{
    w1 = 0.3 * rng.standard_normal((2, 16))
    b1 = frozen[np.zeros(16)]
    w2 = 0.3 * rng.standard_normal((16, 3))
    b2 = np.zeros(3)
} as weights:
    forward = $ |> $ @ w1 + b1 |> np.maximum(0.0, $) |> $ @ w2 + b2
    batched = vmap(forward)
    objective = lambda: batched(X) |> softmax_ce($, Y)
    for _ in range(300):
        value, grads = weights.grad(objective)
        weights.step(grads, 0.5)
    b1_after = weights["b1"].value
    logits = batched(X)

print("final loss        :", round(float(value), 4))
print("b1 stayed all-zero:", bool(np.all(b1_after == 0.0)))
print("train accuracy    :", np.mean(np.argmax(logits, axis=1) == labels))

final loss        : 0.0008
b1 stayed all-zero: True
train accuracy    : 1.0


## Beyond the basics — a small layer toolkit

The fancier models below reuse a few **plain helpers** — `relu`, `sigmoid`,
`layer_norm`, and scaled dot-product `attention`. None has a hand-written gradient:
each is an ordinary numpy function that pycograd instruments on demand, so the
`np.exp` / `np.max` / `@` inside it differentiates whenever a tracer flows through —
a `Var` under `grad`, or a `vmap` batch tracer here.

We also package the training loop as `train(...)` (the same loop as above) plus a
helper that drives a *stateful* optimizer like Adam in place.

In [8]:
import numpy as np
from pycograd import Adam, Param, cosine_decay

# --- layers: just numpy; differentiated by on-demand instrumentation ---------
def relu(z):     return np.maximum(0.0, z)
def sigmoid(z):  return 1.0 / (1.0 + np.exp(-z))

def softmax_last(z):                       # softmax over the last axis (stable)
    z = z - np.max(z, axis=-1, keepdims=True)
    e = np.exp(z)
    return e / np.sum(e, axis=-1, keepdims=True)

def cross_entropy(logits, onehot):         # works for a single (1-D) example too
    return -np.sum(onehot * np.log(softmax_last(logits) + 1e-12))

def layer_norm(x, gamma, beta, eps=1e-5):  # normalize each row, then scale + shift
    mu = np.mean(x, axis=-1, keepdims=True)
    xc = x - mu
    var = np.mean(xc * xc, axis=-1, keepdims=True)
    return xc / np.sqrt(var + eps) * gamma + beta

def attention(q, k, v):                    # scaled dot-product attention
    scores = (q @ k.T) * (q.shape[-1] ** -0.5)
    return softmax_last(scores) @ v

def self_attention(x, Wq, Wk, Wv):         # project x into q/k/v, then attend
    return attention(x @ Wq, x @ Wk, x @ Wv)

def mean_pool(x):  return np.mean(x, axis=0)

# --- training: the same loop as the basic examples, packaged -----------------
def assign_(weights, updated):
    """Copy an optimizer's returned values back into the live params, in place,
    so the ambient `Weight` proxies see the update on the next forward."""
    for k in weights:
        if isinstance(weights[k], Param):
            weights[k].value = updated[k].value

def train(weights, objective, steps, opt):
    """Run `steps` updates. `opt` is either a float lr (in-place SGD via
    weights.step) or an Optimizer instance such as Adam. Returns (first, last) loss."""
    first = last = None
    for _ in range(steps):
        value, grads = weights.grad(objective)
        first = float(value) if first is None else first
        last = float(value)
        if isinstance(opt, float):
            weights.step(grads, opt)                       # plain in-place SGD
        else:
            assign_(weights, opt.step(weights, grads))     # stateful optimizer
    return first, last

## Example 4 — a highway network

A [highway layer](https://arxiv.org/abs/1505.00387) learns a per-unit gate `t` and
mixes a nonlinear transform with a straight-through copy of its input:

$$y = t \odot \mathrm{relu}(xW_h + b_h) + (1 - t) \odot x.$$

That shape is a natural fit for pipescript's **`fork[...]`**, which sends the piped
value down several branches and collects the results into a tuple. `fork[$, ..., ...]`
forks the input into the **carry** (`$`, untouched), the **transform** `H(x)`, and the
**gate** `t`; the tuple pipe `*|>` then splats that triple into the next stage, where
`($x, $h, $t)` names the branches (positionally, matching the fork order) and the
combine refers to them by name. We project the 2-D input to the hidden width, stack two
highway layers, then read out three class logits — and `vmap` runs that whole
per-example forward (`fork`, gates and all) over the `(N, 2)` batch in one pass.

In [9]:
with params{
    Wp = 0.5 * rng.standard_normal((2, 16))     # input projection
    bp = np.zeros(16)
    Wh1 = 0.5 * rng.standard_normal((16, 16))   # highway layer 1
    bh1 = np.zeros(16)
    Wt1 = 0.5 * rng.standard_normal((16, 16))
    bt1 = -1.0 * np.ones(16)                    # gate biased toward "carry"
    Wh2 = 0.5 * rng.standard_normal((16, 16))   # highway layer 2
    bh2 = np.zeros(16)
    Wt2 = 0.5 * rng.standard_normal((16, 16))
    bt2 = -1.0 * np.ones(16)
    Wo = 0.5 * rng.standard_normal((16, 3))     # readout
    bo = np.zeros(3)
} as weights:
    # each highway layer: fork into (carry, transform, gate), then combine by name
    highway1 = ($ |> fork[$, $ @ Wh1 + bh1 |> relu, $ @ Wt1 + bt1 |> sigmoid]
                  *|> ($x, $h, $t) *|> $h * $t + $x * (1 - $t))
    highway2 = ($ |> fork[$, $ @ Wh2 + bh2 |> relu, $ @ Wt2 + bt2 |> sigmoid]
                  *|> ($x, $h, $t) *|> $h * $t + $x * (1 - $t))
    forward = ($ |> $ @ Wp + bp |> relu |> highway1 |> highway2 |> $ @ Wo + bo)
    batched = vmap(forward)
    objective = lambda: batched(X) |> softmax_ce($, Y)
    first, last = train(weights, objective, 300, 0.3)
    logits = batched(X)

print("loss %.4f -> %.4f" % (first, last))
print("train accuracy:", np.mean(np.argmax(logits, axis=1) == labels))

loss 1.8448 -> 0.0007
train accuracy: 1.0


## Example 5 — training with Adam

`weights.grad(objective)` only computes gradients, so *any* optimizer can consume
them. Here we swap in-place SGD for **Adam**. Same `vmap`'d highway model, far fewer
steps to a lower loss.

In [10]:
with params{
    Wp = 0.5 * rng.standard_normal((2, 16))
    bp = np.zeros(16)
    Wh = 0.5 * rng.standard_normal((16, 16))
    bh = np.zeros(16)
    Wt = 0.5 * rng.standard_normal((16, 16))
    bt = -1.0 * np.ones(16)
    Wo = 0.5 * rng.standard_normal((16, 3))
    bo = np.zeros(3)
} as weights:
    highway = ($ |> fork[$, $ @ Wh + bh |> relu, $ @ Wt + bt |> sigmoid]
                 *|> ($x, $h, $t) *|> $h * $t + $x * (1 - $t))
    forward = ($ |> $ @ Wp + bp |> relu |> highway |> $ @ Wo + bo)
    batched = vmap(forward)
    objective = lambda: batched(X) |> softmax_ce($, Y)
    first, last = train(weights, objective, 120, Adam(lr=0.05))
    logits = batched(X)

print("loss %.4f -> %.4f  (120 Adam steps)" % (first, last))
print("train accuracy:", np.mean(np.argmax(logits, axis=1) == labels))

loss 1.5051 -> 0.0000  (120 Adam steps)
train accuracy: 1.0


## Example 6 — a self-attention sequence classifier

Now the inputs are short **sequences** (each a `length × d` matrix). A self-attention
layer lets every position attend to every other — $\mathrm{softmax}(QK^\top/\sqrt{d})\,V$
— before we mean-pool over positions and read out a class.

**This is where `vmap` earns its keep.** In the un-batched demo the objective ran a
Python `for` loop over examples, summing the per-example loss. Here `forward1` is
written for a **single** `length × d` sequence, and `vmap(forward1)` maps it over a
stacked `(B, length, d)` batch in one pass — the loop is gone.

In [11]:
# synthetic sequences: two classes separated by a per-token mean shift
def make_sequences(n_per=12, L=4, d=8, seed=7):
    g = np.random.default_rng(seed)
    seqs, labels_ = [], []
    for cls, shift in enumerate((-0.7, 0.7)):
        for _ in range(n_per):
            seqs.append(g.normal(shift, 0.6, size=(L, d)))
            labels_.append(cls)
    return np.stack(seqs), np.array(labels_)   # (B, L, d) batch

S, S_labels = make_sequences()
S_oh = np.eye(2)[S_labels]
d_model = S.shape[-1]

with params{
    Wq = 0.3 * rng.standard_normal((d_model, d_model))
    Wk = 0.3 * rng.standard_normal((d_model, d_model))
    Wv = 0.3 * rng.standard_normal((d_model, d_model))
    Wc = 0.3 * rng.standard_normal((d_model, 2))   # classifier head
    bc = np.zeros(2)
} as weights:
    # forward1 acts on ONE (L, d) sequence; vmap maps it over the (B, L, d) batch
    forward1 = ($ |> self_attention($, Wq, Wk, Wv) |> mean_pool($) |> $ @ Wc + bc)
    batched = vmap(forward1)                       # (B, L, d) -> (B, 2)
    objective = lambda: batched(S) |> softmax_ce($, S_oh)
    first, last = train(weights, objective, 150, Adam(lr=0.05))
    preds = np.argmax(batched(S), axis=1)

print("loss %.4f -> %.4f" % (first, last))
print("train accuracy:", np.mean(preds == S_labels))

loss 0.7085 -> 0.0000
train accuracy: 1.0


## Example 7 — a Transformer encoder block

The capstone composes everything: self-attention with an output projection, a residual
connection and `layer_norm`, then a position-wise feed-forward network with its own
residual + norm — one standard Transformer encoder block, written for a single
sequence and `vmap`'d over the batch. We train it with Adam and a **cosine-decayed**
learning rate (a schedule is just a `callable(step) -> lr`).

In [12]:
def transformer_block(x, Wq, Wk, Wv, Wo, g1, beta1, W1, c1, W2, c2, g2, beta2):
    a = self_attention(x, Wq, Wk, Wv) @ Wo
    x = layer_norm(x + a, g1, beta1)            # residual + norm
    return x @ W1 + c1 |> relu |> $ @ W2 + c2 |> layer_norm(x + $, g2, beta2)
    # ff = relu(x @ W1 + c1) @ W2 + c2            # position-wise feed-forward
    # return layer_norm(x + ff, g2, beta2)        # residual + norm

d_ff = 16
with params{
    Wq = 0.3 * rng.standard_normal((d_model, d_model))
    Wk = 0.3 * rng.standard_normal((d_model, d_model))
    Wv = 0.3 * rng.standard_normal((d_model, d_model))
    Wo = 0.3 * rng.standard_normal((d_model, d_model))   # attention output proj
    g1 = np.ones(d_model)
    beta1 = np.zeros(d_model)
    W1 = 0.3 * rng.standard_normal((d_model, d_ff))      # FFN up
    c1 = np.zeros(d_ff)
    W2 = 0.3 * rng.standard_normal((d_ff, d_model))      # FFN down
    c2 = np.zeros(d_model)
    g2 = np.ones(d_model)
    beta2 = np.zeros(d_model)
    Wc = 0.3 * rng.standard_normal((d_model, 2))         # classifier head
    bc = np.zeros(2)
} as weights:
    forward1 = ($ |> transformer_block($, Wq, Wk, Wv, Wo, g1, beta1, W1, c1, W2, c2, g2, beta2)
                  |> mean_pool($) |> $ @ Wc + bc)
    batched = vmap(forward1)                       # (B, L, d) -> (B, 2)
    objective = lambda: batched(S) |> softmax_ce($, S_oh)
    schedule = cosine_decay(0.05, total_steps=200)
    first, last = train(weights, objective, 200, Adam(lr=schedule))
    preds = np.argmax(batched(S), axis=1)

print("loss %.4f -> %.4f  (Adam + cosine schedule)" % (first, last))
print("train accuracy:", np.mean(preds == S_labels))

loss 1.6512 -> 0.0000  (Adam + cosine schedule)
train accuracy: 1.0


## More

`vmap` is one trace level in pycograd's interpreter stack, so it composes the other
way too: `vmap(grad(f))` gives **per-sample** gradients (one gradient per example,
stacked over the batch) and `vmap(vmap(f))` nests batch axes. The bundled demos train
from scratch and are gradient-checked against finite differences:

```
python -m pycograd.examples
```